In [102]:
import numpy as np
import scipy.linalg as sp

## Class for Topological space

In [103]:
class Top:
    ''' A class for representing a finite topological space. 
        ------------
        Attributes |
        ----------------------------------------------------------------
        points: a set {x | x in X } of the points of the top. space
                The elements should all be hashable

        opens: a dictionary of the basic open sets. It is of the form
        { x : { y | x <= y} } where x <= y means x in closure({y})

        closures:   dictionary of the closures of each point
                    { x : cl({x}) }
                    Starts as empty because computing all is O(n^2)
                    Run getClosures() method if all are needed. Else,
                    just call X.cl(x) or X[x] for closure of 1 point
        ----------------------------------------------------------------
    
    '''
    def __init__(self, data, discrete=False, boolean=False):
        # can pass a networkx graph as the data
        # import networkx as nx

        if type(data) == dict: # dict of open sets {x: {y | y <= x} }
            self.opens = data
            self.points = set(data)
        
        elif type(data) == int:
            assert data >= 0, 'Topology of integer can\'t be negative'
            assert (discrete and boolean)==False, ('boolean lattice is '
                                                    'not discrete')
            self.points = {n for n in range(data)}            
            if not discrete: 
                if not boolean: # order linearly
                    self.opens = {n: {k for  k in range(n+1)} 
                                for n in range(data)}
                else: # make power set on n elements <= under inclusion
                    # use binary expansion to encode subsets
                    pts = range(1 << data)
                    self.points = {pt for pt in pts}
                    self.opens = {pt: {x for x in pts if x & pt == x}
                                  for pt in pts}

            else: # discrete topology
                self.opens = {n: {n} for n in range(data)}

        ''' optional functionality with networkx directed graphs, but I
        currently am not using this, so I don't want to bother importing '''

        # elif type(data) == nx.DiGraph:
        #     opens = {}
        #     for node in data.nodes:
        #         opens[node] = nx.descendants(data,node) | {node}
        #     self.opens = opens
        #     self.points = set(data.nodes)
            
        self.closures = {} # can specify closures and ordering
        self.ordering = None # but left empty until needed
        self.maxs = None # max/minimal elements w.r.t. partial ordering
        self.mins = None # (must be T_0 space to make sense)
        self.T0 = None # has been checked to be T0


    def __repr__(self):
        return str(self.opens)
    
    def __len__(self):
        return len(self.points)
    
    def __contains__(self, x):
        return x in self.points
    
    def __iter__(self):
        return iter(self.points)

    def __call__(self, x):
        # Top(x) returns the same as self.opens[x] = {y | y <= x}
        return self.opens[x]
        
    def __getitem__(self, x):
        # Top[x] returns the same as Top.cl(x) or Top.closures[x]
        # This convention mirrors the notation (a,b) for open interval 
        # and [a,b] for closed interval
        try:
            return self.closures[x]
        except:
            self.closures[x] = self.cl(x)
            return self.closures[x]
        
    def __mul__(self, other):
        # product topology
        return self.product(other)
    
    def __add__(self, other):
        # disjoint union topology
        if not self.points & other.points: # already disjoint
            return Top(self.opens | other.opens)
        else: # force them to be disjoint
            point0 = Top({0: {0}})
            point1 = Top({1: {1}})
            return Top((point0 * self).opens | (point1 * other).opens)
        
    def __eq__(self, other):
        return self.opens == other.opens
    
    def __le__(self, other):
        # is a subspace
        if self.points & other.points != self.points:
            return False # isn't a subset
        for point in self.points:
            if other(point) & self.points != self(point):
                return False # not the same open sets
        return True
    
    def __lt__(self, other):
        # proper subspace
        return (not self == other) and self <= other
    
    def __ge__(self, other):
        # contains other as a subspace
        return other <= self
    
    def __gt__(self, other):
        # proper super space
        return (not self == other) and self >= other
    
    def __truediv__(self, subspace):
        # quotient by a subspace
        assert self >= subspace, 'Quotient defined for a subspace'
        quotient = set(self.points - subspace.points)

        # the quotient projection
        def pi(x, pt): 
            if x in self.points - subspace.points:
                return x
            elif x in subspace.points:
                return pt
            else:
                raise ValueError('projection map domain error')
        
        # adjoin a new point not in X \ A, make sure key isn't taken
        pt = '*'; idx = 0
        while pt in quotient:
            idx += 1
            pt = '*' + str(idx)
        # now make the open sets for each x in X \ A
        opens = {x: {pi(y, pt) for y in self(x)} for x in quotient}
        # add in the open set around the new extra point 
        opens[pt] = set()
        for a in subspace.points:
            opens[pt] |= {pi(x, pt) for x in self(a)}
        return Top(opens)
    
    def __pow__(self, other):
        return other.hom(self)

    def __invert__(self):
        return self.op()

    def cl(self, x):
        # returns the closure of the singleton {x}
        assert x in self.points,\
        f'{x} is not a point in the topological space'
        close = {y for y in self if x in self(y)}
        # close = { y | x <= y } = { y | x in self.opens[y] }
        return close
        
    def getClosures(self):
        # populates Top.closures for all elements
        for x in self:
            try: # don't bother if it's already computed
                self.closures[x]
            except:
                self.closures[x] = self.cl(x)

    def isleq(self, x, y):
        # partial order defined by x<=y iff x in self.opens[y]
        # or equivalently iff y in self.closures[x]
        return x in self(y)
    
    def isT0(self):
        # True iff self.opens[x] = self.opens[y] implies x = y
        if self.T0 == None:
            checked = set()
            for x in self:
                for y in self.points - (checked | {x}):
                    if self(x) == self(y):
                        self.T0 = False
                        return False
                checked |= {x} # don't check U_x=U_y and U_y=U_x separately
            self.T0 = True
        return self.T0
    
    def isT1(self):
        # for finite spaces, this is the same as being discrete
        for x in self:
            if self(x) != {x}:
                return False
        return True
    
    def subspace(self, subset):
        # subspace topology
        assert subset & self.points == subset, 'needs to be a subset'
        return Top({x: self(x) & subset for x in subset})
    
    def product(self, other):
        # cartesian product
        def prod(set1, set2):
            return [(x, y) for x in set1 for y in set2]
        ord1 = self.order()
        ord2 = other.order()
        points = prod(ord1, ord2)
        opens = {point : set(prod(self(point[0]), other(point[1]))) \
                 for point in points}
        return Top(opens)
    
    def op(self):
        # reverse the ordering
        self.getClosures()
        op_opens = self.closures
        X_op = Top(op_opens)
        X_op.closures = self.opens
        X_op.maxs = self.mins
        X_op.mins = self.maxs
        if self.ordering:
            op_order = list(self.ordering).copy()
            op_order.reverse()
            X_op.ordering = tuple(op_order)
        return (X_op)
    
    def order(self, update=True):
        # returns a topological ordering of self.points, meaning that if
        # x <= y, then x will appear first in the list (but converse may 
        # be false if x and y aren't comparable)

        nodes = {x: {'start':0, 'end':0, 'parent':None, 'visited':False}
                 for x in self.points} # nodes for depth first search
        time = 0 
        def dfsVisit(pt): # one iteration of depth first search
            nonlocal time
            time += 1
            node = nodes[pt]
            node['start'] = time
            node['visited'] = True

            for child in sorted(list(self[pt]), key=hash, reverse=True): # pt <= child
                childnode = nodes[child]
                if childnode['visited'] == False:
                    childnode['parent'] = node
                    dfsVisit(child)
            time += 1
            node['end'] = time
            node['visited'] = 'done'

        # now iterate over all points 
        ordered_pts = sorted(list(self.points), key=hash, reverse=True)
        for pt in ordered_pts:
            if nodes[pt]['visited'] == False:
                dfsVisit(pt)

        # after depth first search, order the points by finish time
        top_order = sorted(nodes, reverse=True,
                           key=lambda x: nodes[x]['end'] )
        if update:
            self.ordering = tuple(top_order)
        return tuple(top_order)
    
    def to(self, other, inj=False):
        ''' returns a list of continuous fnunctions self -> other. Each
        function is represented as a dictionary {x: f(x)}.
        X.to(Y) returns [{x: f(x)} for f: X --> Y continuous]'''
        P_ordered = self.order()          # topologically sorted points
        results = []

        # helper function to iteratively modify an existing function
        def backtrack(index, current_map):  
            if index == len(P_ordered): # complete function
                results.append(current_map.copy())
                return 
            x = P_ordered[index]
            for q in other.points: # potential values for f(x)
                valid = True # make sure (x <= y) ==> ( f(x) <= f(y) )
                for y in current_map:
                    fy = current_map[y]
                    if y in self.opens[x] and fy not in other.opens[q]:
                        valid = False
                        break
                if valid:
                    current_map[x] = q # assign f(x) = q
                    backtrack(index + 1, current_map) # progress 1 step
                    # reached end of valid mapping
                    del current_map[x] # remove last assignment, iterate

        backtrack(0, {})
        return results
    
    def hom(self, other, domain_order=False):
        # X.hom(Y) returns the topological space Hom(X, Y) with the com-
        # pact-open topology: f <= g iff f(x) <= g(x) for all x
        funcs = self.to(other)
        opens = {}
        order = self.ordering if self.ordering else self.order()
        def funcTuple(func):
            # use top ordering to write func as (f(x_0), ..., f(x_n))
            # so that it is hashable            
            return tuple([func[x] for x in order])

        for f1 in funcs:
            key = funcTuple(f1)
            opens[key] = {funcTuple(f2) for f2 in funcs
                          if all(f2[x] in other.opens[f1[x]] 
                                        for x in self.points)    
                            }
        if domain_order:
            return order, Top(opens)
        return Top(opens)
    
    def relabel(self):
        # returns a homeomorphic top space with integer labels
        order = self.ordering if self.ordering else self.order()
        opens = {n:{m for m in range(len(self)) 
                    if order[m] in self(order[n])}
                for n in range(len(self))}
        return Top(opens)
    
    def succ(self, x, ordered=False):
        ''' Given point x in X, returns the immediate successors 
            {z in X | x < z and (x<y<=z ==> y=z) }  '''
        assert x in self.points, f'{x} is not a point in the topological space'
        succ = self[x] - {x} # remove x from the set of successors
        for y in succ.copy(): # if y > x
            if y not in succ: # y may have been removed proviously
                continue
            for z in (self[y] - {y}) & succ: # and z > y
                succ -= {z} # z can't be a successor of x
        if ordered:
            succ = self.sort(succ)       
        return succ
    
    def pred(self, x, ordered=False):
        ''' Given point x in X, returns the immediate predecessors 
            {z in X | z < x and (y<z<=x ==> y=z) }  '''
        assert x in self.points, f'{x} is not a point in the topological space'
        pred = self(x) - {x} # remove x from the set of successors
        for y in pred.copy(): # if y > x
            if y not in pred: # y may have been removed proviously
                continue
            for z in (self(y) - {y}) & pred: # and z > y
                pred -= {z} # z can't be a successor of x
        if ordered:
            pred = self.sort(pred)       
        return pred


    def getMaxs(self, checkT0=False):
        '''Returns the set of maximal elements wrt the partial ordering.
        Needs to be T_0 space otherwise can have infinite chains. '''
        if checkT0: # O(n^2) to check, so better to avoid if possible            
            assert self.isT0(), ('maximal elements aren\'t defined for pre-'
            'orders which aren\'t partial orders') 
        if self.maxs == None: # not already computed
            self.maxs = {x for x in self if self[x] == {x}}
        return self.maxs
    
    def getMins(self, checkT0=False):
        '''Returns the set of maximal elements wrt the partial ordering.
        Needs to be T_0 space otherwise can have infinite chains. '''
        if checkT0: # O(n^2) to check, so better to avoid if possible            
            assert self.isT0(), ('maximal elements aren\'t defined for pre-'
            'orders which aren\'t partial orders') 
        if self.mins == None: # not already computed
            self.mins = {x for x in self if self(x) == {x}}
        return self.mins

    def maxChains(self, checkT0=False):
        ''' Returns a list of the maximal chains in a T0 top. sp. '''
        if checkT0:
            assert self.isT0(), ('maximal chains aren\'t defined for pre-'
            'orders which aren\'t partial orders')
        
        order = self.order() # for consistency of ordering
        def index(pt): # pass as key to sorted() to sort by top ordering
            for i, x in enumerate(order):
                if x == pt:
                    return i
        maxs = sorted(self.getMaxs(), key=index)
        mins = sorted(self.getMins(), key=index) 
        chains = []
        

        def extendChain(chain):
            # given a non-empty chain (as list), extend it further is possible
            curr_max = chain[-1]
            bigger = sorted(self.succ(curr_max), key=index)
            if curr_max in maxs: # already maximal, add to list
                chains.append(tuple(chain))
                return
            for pt in bigger:
                new_chain = chain.copy()
                new_chain.append(pt)
                extendChain(new_chain)
        
        # extend all chains starting at minimal elements
        for min in mins:
            extendChain([min])
        return chains # will be ordered lexographically
        
    def grading(self, ordered=False):
        ''' Returns a dictionary {n : {points n levels up from bottom}}
        grading[0]={minimals}, grading[1]={x | y < x ==> y minimal}, etc
        
        If (X, <=) comes from a simplicial complex wrt inclusion, then
        X.grading()[n] is the set of n-faces. If not, then the grading 
        isn't necessarily unique, so '''
        grade = {}
        level = 0
        grade[0] = self.getMins()
        remaining = self.points - grade[0]
        while remaining != set():
            level += 1
            grade[level] = set()
            for pt in grade[level - 1]:
                grade[level] |= self.succ(pt)
            remaining -= grade[level]
        if ordered:
            order = self.order()
            def index(pt): # pass as key to sorted() to sort by top ordering
                for i, x in enumerate(order):
                    if x == pt:
                        return i
            for n in grade: # return ordered tuples instead of sets
                in_order = sorted(list(grade[n]), key=index)
                grade[n] = tuple(in_order)
        return grade

    def index(self, pt): # pass as key to sorted() to sort by top ordering
        order = self.ordering if self.ordering else self.order()
        for i, x in enumerate(order):
            if x == pt:
                return i
    
    def sort(self, subset, check_subset=True):
        ''' given a subset A <= X, returns A in sorted order accoring to 
        the topological ordering of X. '''
        if check_subset:
            assert subset & self.points == subset, 'needs to be a subset'
        pts = list(subset)
        return tuple(sorted(pts, key=self.index))


### Here's a separate class for a simplicial complex

In [ ]:
''' First, define some funtions for integer to bit conversion '''

def numtobits(n : int) -> list[int]:
    # given n, return a list of which bits are 1 in the binary
    index = [i for i in range(n.bit_length()) if (n >> i) & 1]
    return index
def bitstonum(bits : list[int]) -> int:
    # given array of which bits are 1 in binary, return the integer
    return sum([1 << bit for bit in bits])

def subfaces(face : int, nonempty : bool = True) -> set[int]:
        # given face, return all subfaces
        pts = numtobits(face)
        start = 1 if nonempty else 0
        subs = [numtobits(sub) for sub in range(start, 1 << len(pts))]
        subface = [[pts[i]   for i in sub] for sub in subs]
        return {bitstonum(bits) for bits in subface}  

class Simp:
    ''' Encodes an abstract simplicial complex using bitwise operations 
    on integers. For example, the line {{x0}, {x1}, {x0, x1}} would be 
    {1,10,11} in binary. Only need to feed in the maximal simplices, all
    others can be infered. Right now only works for simplices with less
    than 65 vertices. '''

    
    def __init__(self, max_faces : int | list[int]) -> None:
        # determine which points occur in the complex
        total = 0
        for face in max_faces:
             total |= face
        self.points = {1 << bit for bit in numtobits(total)}
        max_faces = [max_faces] if type(max_faces) == int else max_faces


        self.dim : int = max([face.bit_count() for face in max_faces])
        self.maxs = max_faces
        self.faces : dict[set[int]] | None = None # compute only if necessary
        
          
    def getFaces(self) -> dict[set[int]]: # compute all faces from maximal ones
        if self.faces: 
            return self.faces
        ''' a bit faster if these funcs are in local scope'''
        def numtobits(n : int) -> list[int]:
            # given n, return a list of which bits are 1 in the binary
            index = [i for i in range(n.bit_length()) if (n >> i) & 1]
            return index
        def bitstonum(bits : list[int]) -> int:
            # given array of which bits are 1 in binary, return the integer
            return sum([1 << bit for bit in bits])
        def subfaces(face : int, nonempty : bool = True) -> set[int]:
                # given face, return all subfaces
                pts = numtobits(face)
                start = 1 if nonempty else 0
                subs = [numtobits(sub) for sub in range(start, 1 << len(pts))]
                subface = [[pts[i]   for i in sub] for sub in subs]
                return {bitstonum(bits) for bits in subface}  
        faces : list[set] = [subfaces(max_face) for max_face in self.maxs]
        self.faces : dict = {-1: set(), 0: self.points} # {} is -1 simplex :)
        for n in range(1, self.dim):
             self.faces[n] = {k for k in set.union(*faces) 
                              if k.bit_count()==n+1}      
        return self.faces # {n: {simplices of dimension n}}
            
    def bd(self, simplex : int, orient : bool = False) -> set[int]:
         ''' Returns the maximal subfaces of the simplex. For now I don't
         care about doing the orientation'''

         bits = [i for i in range(simplex.bit_length()) if (simplex >> i & 1)]
         if not orient:
            return {simplex - (1 << bit) for bit in bits}
         
         else: # if oriented, the orientation is encoded by negative signs
              return {(-1) ** i * (simplex - (1 << bit)) 
                      for i, bit in enumerate(bits)}


In [ ]:
def rref2(matrix):
    ''' Returns (A, p), where A is the reduced row echelon form over F_2
    and p is the indices of the pivot columns'''
    A = np.array(matrix, dtype=bool)
    n, m = A.shape
    A_rref = A.copy()

    row = 0
    pivot_cols = []

    # Row reduction (mod 2, using XOR logic)
    for col in range(m):
        # Find pivot
        pivot_row = None
        for r in range(row, n):
            if A_rref[r, col]:
                pivot_row = r
                break
        if pivot_row is None:
            continue

        # Swap pivot row into position
        if pivot_row != row:
            A_rref[[row, pivot_row]] = A_rref[[pivot_row, row]]

        # Eliminate other rows
        for r in range(n):
            if r != row and A_rref[r, col]:
                A_rref[r] ^= A_rref[row]  # XOR for row elimination

        pivot_cols.append(col)
        row += 1
        if row == n:
            break
        
    return A_rref, np.array(pivot_cols)

def im2(matrix):
    ''' Returns a basis for the image of a matrix over F2'''
    A, pivots = rref2(matrix)
    return np.transpose(A[:, pivots])

def ker2(matrix):
    ''' Returns a basis for the kernel of a matrix over F2'''
    A, pivots = rref2(matrix)
    m = A.shape[1] # number of columns
    
    # Kernel basis: construct for each free variable
    free_vars = [j for j in range(m) if j not in pivots]
    kernel_basis = []

    for free in free_vars:
        vec = np.zeros(m, dtype=bool)
        vec[free] = True
        for i in reversed(range(len(pivots))):
            col = pivots[i]
            row_vals = A[i]
            sum_ = False
            for j in free_vars:
                sum_ ^= row_vals[j] and vec[j]
            vec[col] = sum_
        kernel_basis.append(vec)
    return np.array(kernel_basis)

def extendBasis2(subspace, space):
    '''Given a basis b_1 of subspace and a basis b_2 of the whole space,
    returns vectors b_3=[v_1, ..., v_n] such that b_1 U b_3 is a basis
    of the whole space. Equivalently, b_3 is representatives for a basis
    of the quotient space, which is isomorphic to the orthogonal complement
    subspace^perp

    The bases should be given as [v_j] where each v_j is a row vector
      '''
    # first make an augmented matrix whose columns are A = [ b_1 | b_2 ]
    U = np.transpose(np.array(subspace))
    V = np.transpose(np.array(space))
    dim_subspace = U.shape[1]
    A = np.hstack((U, V))
    pivots = rref2(A)[1]
    orth_pivots = pivots[pivots >= dim_subspace]
    return np.transpose(A[:, orth_pivots])

def homology2(complex : dict[np.ndarray]):
    # todo



In [ ]:
a  = np.array([[1,1,0],[1,0,1],[0,1,1]], dtype='uint8')
b = a[0:0,:]; 
c = np.identity(3, dtype='uint8')
extendBasis2(b, c)


array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]], dtype=uint8)

In [ ]:
def homology(X : Top):
    ''' Assume X is already simplicial complex with <= being inclusion.
    For now i am just doing Z_2 coefficients for simplicity'''
    import numpy as np
    order = X.order()
    faces = X.grading(ordered=True)
    faces = {n: np.array(faces[n]) for n in faces}
    n = len(faces)
    numfaces = [len(faces[j]) for j in range(n)]
    # dictionary delta = boundary maps between faces {n : delta_n}
    delta = {0: np.zeros((1,numfaces[0]), dtype=bool),
             n: np.zeros((numfaces[n-1], 0))} 
    cycle_sets = {0: [{v} for v in faces[0]]} # every vertex maps to 0
    bdry_sets = {0: [set()]} # which means 0 image
    for dim in range(1, n):
        dom, rang = numfaces[dim], numfaces[dim-1]
        simplex, face = faces[dim], faces[dim-1]

        delta[dim] = np.array([[face[j] in X.pred(simplex[i]) 
                                for i in range(dom)] 
                                for j in range(rang)], dtype=bool)
        cycles = ker2(delta[dim])
        bdrys =  im2(delta[dim+1])
        cycle_sets[dim] = [set(faces[dim][cycle]) for cycle in cycles]
        bdry_sets[dim] = [set(faces[dim][bdry]) for bdry in bdrys]
        
    
    return cycle_sets, bdry_sets
        # try scipy.linalg.(orth, null_space and/or qr)
        
    

In [ ]:
a = np.array([0,1,2]); b = np.array([[True, False, True], [True, False, False]])
c = np.array([True, False, True])
a[c]
[{v} for v in a]

[{0}, {1}, {2}]

In [ ]:
W = Top({0:{0},1:{1},2:{2},3:{3, 0, 1},4:{4, 0, 1},5:{5, 1, 2},6:{6, 1, 2}})
homology(W)


({0: [{0}, {1}, {2}], 1: [{3, 4}, {5, 6}]}, {0: [set()], 1: []})

In [ ]:
a = np.random.randint(2,size=(1000,1000))
%timeit im_ker_basis(a)
%timeit rref2(a)
%timeit im2(a)


368 ms ± 4.46 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
365 ms ± 4.51 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
375 ms ± 6.49 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
X = Top({0:{0,1,3}, 1:{1}, 2:{1,2,3}, 3:{3}})
A = X.subspace({0,3})
X.getClosures()
X.getMaxs(True)
X.getMins()
X.mins, X.maxs

({1, 3}, {0, 2})

In [ ]:
N =4
Nsimp = Top(N, boolean=True).subspace({n for n in range(1,2**N)})

homology(Nsimp)

({0: array([[False, False, False, False]]),
  1: array([[ True,  True, False,  True, False, False],
         [ True, False,  True, False,  True, False],
         [False,  True,  True, False, False,  True],
         [False, False, False,  True,  True,  True]]),
  2: array([[ True,  True, False, False],
         [ True, False,  True, False],
         [ True, False, False,  True],
         [False,  True,  True, False],
         [False,  True, False,  True],
         [False, False,  True,  True]]),
  3: array([[ True],
         [ True],
         [ True],
         [ True]])},
 {0: (1, 2, 4, 8), 1: (3, 5, 6, 9, 10, 12), 2: (7, 11, 13, 14), 3: (15,)})

In [ ]:
homology(X)

({0: array([[False, False]]),
  1: array([[ True,  True],
         [ True,  True]])},
 {0: (1, 3), 1: (0, 2)})

In [ ]:
X.grading(), [(x,X.pred(x, True)) for x in X]

({0: {1, 3}, 1: {0, 2}}, [(0, (1, 3)), (1, ()), (2, (1, 3)), (3, ())])

In [ ]:
X.order(), (X.op()).order(), X.grading(), X.grading(ordered=True)

((1, 3, 0, 2), (0, 2, 1, 3), {0: {1, 3}, 1: {0, 2}}, {0: (1, 3), 1: (0, 2)})

In [ ]:
Y = Top({  2:{1,2,3}, 3:{3},1:{1}, 0:{0,1,3}})
Y.order()
Y.relabel()


{0: {0}, 1: {1}, 2: {0, 1, 2}, 3: {0, 1, 3}}

In [ ]:
X.maxChains(), [(x, X.pred(x)) for x in X]

([(1, 0), (1, 2), (3, 0), (3, 2)],
 [(0, {1, 3}), (1, set()), (2, {1, 3}), (3, set())])

In [ ]:
Z =Top({0:{0},1:{0,1},2:{0,1,2,9},3:{3,7,0},4:{4,3,7,0},5:{5,3,7,0},
        6:{6,4,5,3,7,0},7:{7},8:{8}, 9:{9}}) #custom example to test edge cases
Z.maxChains(), Z.order(), Z.grading()

([(0, 1, 2),
  (0, 3, 4, 6),
  (0, 3, 5, 6),
  (7, 3, 4, 6),
  (7, 3, 5, 6),
  (8,),
  (9, 2)],
 (0, 1, 7, 3, 4, 5, 6, 8, 9, 2),
 {0: {0, 7, 8, 9}, 1: {1, 2, 3}, 2: {2, 4, 5}, 3: {6}})

In [ ]:
W = Top({0:{0},1:{1},2:{2},3:{3, 0, 1},4:{4, 0, 1},5:{5, 1, 2},6:{6, 1, 2}})
W.maxChains(), [(x, W.pred(x)) for x in W], W.grading() # join of 2 circles

([(0, 3), (0, 4), (1, 3), (1, 4), (1, 5), (1, 6), (2, 5), (2, 6)],
 [(0, set()),
  (1, set()),
  (2, set()),
  (3, {0, 1}),
  (4, {0, 1}),
  (5, {1, 2}),
  (6, {1, 2})],
 {0: {0, 1, 2}, 1: {3, 4, 5, 6}})

In [ ]:
p = (((1,1),1),1)
(X * X * X * X).succ(p)

{(((0, 1), 1), 1),
 (((1, 0), 1), 1),
 (((1, 1), 0), 1),
 (((1, 1), 1), 0),
 (((1, 1), 1), 2),
 (((1, 1), 2), 1),
 (((1, 2), 1), 1),
 (((2, 1), 1), 1)}

In [ ]:
(X ** Top(4)).relabel()

{0: {0}, 1: {1}, 2: {1, 2}, 3: {1, 3}, 4: {1, 3, 4}, 5: {1, 2, 5}, 6: {1, 2, 5, 6}, 7: {0, 7}, 8: {0, 8, 7}, 9: {0, 8, 9, 7}, 10: {0, 1, 2, 5, 6, 7, 8, 9, 10}, 11: {11, 1, 3, 4}, 12: {0, 12}, 13: {0, 12, 13}, 14: {0, 12, 13, 14}, 15: {0, 1, 3, 4, 11, 12, 13, 14, 15}}

In [ ]:
(Top(5)*Top(5)).grading(ordered=True), (Top(5)*Top(5)).grading(ordered=False), (Top(5)*Top(5)).order()

({0: ((0, 0),),
  1: ((1, 0), (0, 1)),
  2: ((2, 0), (0, 2), (1, 1)),
  3: ((3, 0), (0, 3), (1, 2), (2, 1)),
  4: ((4, 0), (0, 4), (1, 3), (2, 2), (3, 1)),
  5: ((1, 4), (2, 3), (4, 1), (3, 2)),
  6: ((2, 4), (4, 2), (3, 3)),
  7: ((4, 3), (3, 4)),
  8: ((4, 4),)},
 {0: {(0, 0)},
  1: {(0, 1), (1, 0)},
  2: {(0, 2), (1, 1), (2, 0)},
  3: {(0, 3), (1, 2), (2, 1), (3, 0)},
  4: {(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)},
  5: {(1, 4), (2, 3), (3, 2), (4, 1)},
  6: {(2, 4), (3, 3), (4, 2)},
  7: {(3, 4), (4, 3)},
  8: {(4, 4)}},
 ((0, 0),
  (1, 0),
  (0, 1),
  (2, 0),
  (3, 0),
  (4, 0),
  (0, 2),
  (0, 3),
  (0, 4),
  (1, 1),
  (1, 2),
  (1, 3),
  (1, 4),
  (2, 1),
  (2, 2),
  (2, 3),
  (2, 4),
  (3, 1),
  (4, 1),
  (3, 2),
  (4, 2),
  (3, 3),
  (4, 3),
  (3, 4),
  (4, 4)))